# 05. Запасы, риски и бизнес-выводы

В исходных данных нет фактических складских остатков. Поэтому расчеты ниже — сценарный анализ на основе прогноза спроса, ошибки прогноза и простого бизнес-допущения `lead_time_days = 7`. Это не доказанный экономический эффект, а способ выделить товары и рынки для контроля.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.baselines import (
    median_by_weekday,
    moving_average_7,
    moving_average_28,
    naive_last_value,
    seasonal_naive_7,
    seasonal_naive_28,
)
from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.inventory import inventory_policy_table
from src.validation import train_holdout_split

## Модельный прогноз

In [ ]:
lead_time_days = 7
forecast_path = RESULTS_DIR / 'forecast_vs_actual_sample.csv'
if not forecast_path.exists():
    raise FileNotFoundError('Сначала запустите notebook 04, чтобы получить forecast_vs_actual_sample.csv')

model_forecast = pd.read_csv(forecast_path, parse_dates=['sales_date'])
model_policy = inventory_policy_table(model_forecast, lead_time_days=lead_time_days)
model_policy['source'] = 'model'
model_policy.head()

## Baseline для сравнения рисков

In [ ]:
baseline_functions = {
    'naive_last_value': naive_last_value,
    'seasonal_naive_7': seasonal_naive_7,
    'seasonal_naive_28': seasonal_naive_28,
    'moving_average_7': moving_average_7,
    'moving_average_28': moving_average_28,
    'median_by_weekday': median_by_weekday,
}

baseline_metrics_path = RESULTS_DIR / 'baseline_metrics.csv'
if baseline_metrics_path.exists():
    baseline_metrics = pd.read_csv(baseline_metrics_path)
    best_baseline = (
        baseline_metrics[baseline_metrics['metric'].str.lower() == 'wmape']
        .sort_values('value')
        .iloc[0]['baseline']
    )
else:
    best_baseline = 'moving_average_7'

features_path = PROCESSED_DATA_DIR / 'features_lags_rolling.csv'
features = pd.read_csv(features_path, parse_dates=['sales_date'])
baseline_frame = features[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].copy()
baseline_frame['forecast_net_sales_qty'] = baseline_functions[best_baseline](baseline_frame)
_, baseline_holdout = train_holdout_split(baseline_frame.dropna(subset=['forecast_net_sales_qty']), 'sales_date', holdout_days=28)
baseline_policy = inventory_policy_table(baseline_holdout, lead_time_days=lead_time_days)
baseline_policy['source'] = best_baseline
best_baseline, baseline_policy.head()

## Таблица рисков

In [ ]:
inventory_risk_table = pd.concat([model_policy, baseline_policy], ignore_index=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
inventory_risk_table.to_csv(RESULTS_DIR / 'inventory_risk_table.csv', index=False)
inventory_risk_table.head()

## Top-20 с высоким сценарным stockout risk

In [ ]:
top_stockout = (
    model_policy.sort_values(['stockout_risk_rate', 'forecast_error', 'avg_daily_demand'], ascending=False)
    .head(20)
)
top_stockout.to_csv(RESULTS_DIR / 'top_stockout_risk_products.csv', index=False)
top_stockout

## Top-20 с риском избыточного запаса

In [ ]:
top_overstock = (
    model_policy.sort_values(['overstock_risk_rate', 'forecast_bias', 'avg_forecast_sales'], ascending=False)
    .head(20)
)
top_overstock.to_csv(RESULTS_DIR / 'top_overstock_risk_products.csv', index=False)
top_overstock

## Сравнение модели и baseline по рискам

In [ ]:
risk_comparison = (
    inventory_risk_table.groupby('source', as_index=False)
    .agg(
        avg_stockout_risk_rate=('stockout_risk_rate', 'mean'),
        avg_overstock_risk_rate=('overstock_risk_rate', 'mean'),
        avg_forecast_error=('forecast_error', 'mean'),
        avg_forecast_bias=('forecast_bias', 'mean'),
    )
)
risk_comparison

## Выводы

- Расчеты показывают сценарный риск, а не фактический складской дефицит или излишек.
- Товары и рынки с максимальным stockout risk: `[A]`.
- Товары и рынки с максимальным overstock risk: `[B]`.
- Для групп с высоким `forecast_bias` стоит отдельно проверить систематическое завышение или занижение прогноза.
- Предложенное правило reorder point основано на `lead_time_days = 7`; при реальных сроках поставки расчет нужно пересчитать.